Задание 1
Для датафрейма log из материалов занятия создайте столбец source_type по правилам:

если источник traffic_source равен Yandex или Google, то в source_type ставится organic;
для источников paid и email из России ставим ad;
для источников paid и email не из России ставим other;
все остальные варианты берём из traffic_source без изменений.

In [1]:
import pandas as pd

# читаем данные из csv файла
data = pd.read_csv('visit_log.csv', sep=';')
# создаем новый столбец для типа источника
data['source_type'] = ''

# задаем условия и записываем соответствующее значение в новый столбец source_type
data.loc[data['traffic_source'].isin(['yandex', 'google']), 'source_type'] = 'organic'
data.loc[(data['traffic_source'] == 'paid') & (data['region'] == 'Russia'), 'source_type'] = 'ad'
data.loc[(data['traffic_source'] == 'email') & (data['region'] == 'Russia'), 'source_type'] = 'ad'
data.loc[(data['traffic_source'] == 'paid') & ~(data['region'] == 'Russia'), 'source_type'] = 'other'
data.loc[(data['traffic_source'] == 'email') & ~(data['region'] == 'Russia'), 'source_type'] = 'other'

# сохраняем таблицу в csv файл
data.to_csv('new_visit_log.csv', index=False, sep=';')

# выводим первые 5 строк новой таблицы с типом источника
data.head()


,timestamp,visit_id,url,region,user_id,traffic_source,source_type
0,1549980692,e3b0c44298,https://host.ru/3c19b4ef7371864fa3,Russia,b1613cc09f,yandex,organic
1,1549980704,6e340b9cff,https://host.ru/c8d9213a31839f9a3a,Russia,4c3ec14bee,direct,
2,1549980715,96a296d224,https://host.ru/b8b58337d272ee7b15,Russia,a8c40697fb,yandex,organic
3,1549980725,709e80c884,https://host.ru/b8b58337d272ee7b15,Russia,521ac1d6a0,yandex,organic
4,1549980736,df3f619804,https://host.ru/b8b58337d272ee7b15,Russia,d7323c571c,yandex,organic


Задание 2
В файле URLs.txt содержатся URL страниц новостного сайта. Вам нужно отфильтровать его по адресам страниц с текстами новостей. Известно, что шаблон страницы новостей имеет внутри URL конструкцию: /, затем 8 цифр, затем дефис. Выполните действия:

Прочитайте содержимое файла с датафрейм.
Отфильтруйте страницы с текстом новостей, используя метод str.contains и регулярное выражение в соответствие с заданным шаблоном.

In [3]:
import pandas as pd

try:
    # читаем содержимое файла в датафрейм
    urls_df = pd.read_csv('URLs.txt', header=None, names=['url'])

    # фильтруем страницы с текстами новостей
    news_pattern = r'/\d{8}-'
    filtered_news_df = urls_df[urls_df['url'].str.contains(news_pattern)]

    # сохраняем отфильтрованные URL в новый файл
    filtered_news_df.to_csv('Filtered_News_URLs.txt', index=False)
    print("Файл 'Filtered_News_URLs.txt' успешно сохранен.")
    
except Exception as e:
    print(f"Не удалось сохранить файл: {e}")


Файл 'Filtered_News_URLs.txt' успешно сохранен.


Задание 3
Используйте файл с оценками фильмов ml-latest-small/ratings.csv. Посчитайте среднее время жизни пользователей, которые выставили более 100 оценок. Под временем жизни понимается разница между максимальным и минимальным значениями столбца timestamp для данного значения userId.

ОБРАТИТЕ ВНИМАНИЕ!!!
Не совсем понятен контекст задания. Имеется ввиду, что нужно посчитать среднее время жизни каждого пользователя? Или среднее время жизни всех пользователей? В общем, реализовала оба варианта.

In [6]:
# загружаем файл
import pandas as pd
df = pd.read_csv('ratings.csv')

# считаем количество оценок, выставленных каждым пользователем
ratings_count = df.groupby('userId').count()['rating'].reset_index()
ratings_count = ratings_count.rename(columns={'rating': 'ratings_count'})

# считаем среднее время жизни пользователей, которые выставили более 100 оценок
active_users = ratings_count[ratings_count['ratings_count'] > 100]

#считаем среднее время жизни для каждого пользователя
def lifespan(user_ratings):
    return user_ratings['timestamp'].max() - user_ratings['timestamp'].min()

user_lifespans = df.groupby('userId').apply(lifespan).reset_index()
user_lifespans = user_lifespans.rename(columns={0: 'lifespan'})

# среднее значение времени жизни для активных пользователей
result = pd.merge(active_users, user_lifespans, on='userId', how='left')['lifespan'].mean()

print(user_lifespans)


     userId  lifespan
0         1        97
1         2       851
2         3     71198
3         4    203560
4         5      2101
..      ...       ...
666     667      1014
667     668       282
668     669       685
669     670   2162705
670     671  11283984

[671 rows x 2 columns]


C:\Users\Светик\AppData\Local\Temp\ipykernel_8760\1328917650.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_lifespans = df.groupby('userId').apply(lifespan).reset_index()


Задание 4
Дана статистика услуг перевозок клиентов компании по типам (см. файл “Python_13_join.ipynb” в разделе «Материалы для лекции “Продвинутый pandas”» ---- Ноутбуки к лекции «Продвинутый pandas»).
Нужно сформировать две таблицы:

таблицу с тремя типами выручки для каждого client_id без указания адреса клиента;
аналогичную таблицу по типам выручки с указанием адреса клиента.
Обратите внимание, что в процессе объединения таблиц данные не должны теряться.

In [8]:
import pandas as pd

# данные
rzd = pd.DataFrame({
    'client_id': [111, 112, 113, 114, 115],
    'rzd_revenue': [1093, 2810, 10283, 5774, 981]
})
auto = pd.DataFrame({
    'client_id': [113, 114, 115, 116, 117],
    'auto_revenue': [57483, 83, 912, 4834, 98]
})
air = pd.DataFrame({
    'client_id': [115, 116, 117, 118],
    'air_revenue': [81, 4, 13, 173]
})
client_base = pd.DataFrame({
    'client_id': [111, 112, 113, 114, 115, 116, 117, 118],
    'address': ['Комсомольская 4', 'Энтузиастов 8а', 'Левобережная 1а', 'Мира 14', 'ЗЖБИиДК 1',
                'Строителей 18', 'Панфиловская 33', 'Мастеркова 4']
})

# 1. Таблица с тремя типами выручки
try:
    revenue = rzd.merge(auto, on='client_id', how='outer').merge(air, on='client_id', how='outer').fillna(0)
    revenue.to_csv('revenue_without_address.csv', index=False)
    print("Таблица с выручкой успешно создана без адреса клиента.")
    print(revenue.head())  # Вывод первых 5 строк
except Exception as e:
    print(f"Ошибка при создании таблицы без адреса: {e}")

# 2. Таблица с выручкой и адресами
try:
    revenue_with_address = revenue.merge(client_base, on='client_id', how='outer').fillna(0)
    revenue_with_address.to_csv('revenue_with_address.csv', index=False)
    print("Таблица с выручкой успешно создана с адресом клиента.")
    print(revenue_with_address.head())  # Вывод первых 5 строк
except Exception as e:
    print(f"Ошибка при создании таблицы с адресом: {e}")


Таблица с выручкой успешно создана без адреса клиента.
   client_id  rzd_revenue  auto_revenue  air_revenue
0        111       1093.0           0.0          0.0
1        112       2810.0           0.0          0.0
2        113      10283.0       57483.0          0.0
3        114       5774.0          83.0          0.0
4        115        981.0         912.0         81.0
Таблица с выручкой успешно создана с адресом клиента.
   client_id  rzd_revenue  auto_revenue  air_revenue          address
0        111       1093.0           0.0          0.0  Комсомольская 4
1        112       2810.0           0.0          0.0   Энтузиастов 8а
2        113      10283.0       57483.0          0.0  Левобережная 1а
3        114       5774.0          83.0          0.0          Мира 14
4        115        981.0         912.0         81.0        ЗЖБИиДК 1
